## Diffusion U-Net - Explained

This notebook builds a **U-Net architecture** specifically designed for **diffusion models**. Diffusion models generate images by gradually removing noise from random noise, and the U-Net is the neural network that learns to predict what noise to remove at each step.

### What You'll Learn

1. **Basic U-Net Architecture** - Encoder-decoder with skip connections
2. **Timestep Embeddings** - How to tell the network what noise level we're at
3. **Timestep-Conditioned U-Net** - Combining U-Net with timestep information
4. **Sampling Methods** - Different ways to generate images (Euler, Heun, LMS)
5. **Image Quality Evaluation** - Using FID and KID metrics

### Why U-Net for Diffusion?

The U-Net architecture is ideal for diffusion because:
- **Same input/output size**: We input a noisy image and output predicted noise (same dimensions)
- **Skip connections**: Preserve fine details that might be lost in the bottleneck
- **Multi-scale processing**: Capture both local details and global structure

## Google Colab Setup

Run the cells below **once** at the start of each Colab session. They mount Google Drive, set the working directory, install required packages, and clone the `miniai` library.

**GPU note:** Diffusion U-Net training needs a GPU. **Runtime &rarr; GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir('/content/drive/MyDrive/Fast.AI_Colab')
print(os.getcwd())

In [ ]:
!pip install -q fastcore fastai diffusers datasets torcheval accelerate wandb
if not os.path.exists('course22p2'):
    !git clone https://github.com/fastai/course22p2.git
import sys
sys.path.insert(0, os.path.join(os.getcwd(), 'course22p2'))
try:
    import miniai
    print(f'miniai loaded from: {miniai.__file__}')
except ImportError:
    print('ERROR: miniai not found.')

---

*The cells above are Colab-specific setup. The content below is the same as the source notebook (`26_diffusion_unet_explained.ipynb`), unchanged.*

---

---
## Section 1: Environment Setup and Imports

First, we import all the necessary libraries for building and training our diffusion U-Net.

In [ ]:
# GPU Selection (uncomment to select a specific GPU)
import os
# os.environ['CUDA_VISIBLE_DEVICES']='1'  # Use GPU 1 instead of default GPU 0

In [ ]:
# Core libraries for deep learning and data manipulation
import timm               # PyTorch Image Models - pre-trained models
import torch              # PyTorch - main deep learning framework
import random             # Random number generation for reproducibility
import datasets           # HuggingFace datasets for easy data loading
import math               # Mathematical functions (log, sqrt, etc.)
import fastcore.all as fc # Fast.ai core utilities
import numpy as np        # Numerical computing
import matplotlib as mpl  # Plotting configuration
import matplotlib.pyplot as plt  # Plotting

# Diffusion-specific imports
import k_diffusion as K   # Katherine Crowson's diffusion library
import torchvision.transforms as T           # Image transformations
import torchvision.transforms.functional as TF  # Functional transforms
import torch.nn.functional as F              # Neural network functions

# PyTorch utilities
from torch.utils.data import DataLoader, default_collate  # Data loading
from pathlib import Path            # File path handling
from torch.nn import init           # Weight initialization
from fastcore.foundation import L   # Enhanced list class
from torch import nn, tensor        # Neural network modules, tensor creation
from datasets import load_dataset   # Load HuggingFace datasets
from operator import itemgetter     # Get items from dictionaries
from torcheval.metrics import MulticlassAccuracy  # Accuracy metric
from functools import partial       # Partial function application
from torch.optim import lr_scheduler  # Learning rate schedulers
from torch import optim             # Optimizers

# Mini AI library modules (custom implementations)
from miniai.datasets import *    # Dataset utilities
from miniai.conv import *        # Convolution helpers
from miniai.learner import *     # Training loop (Learner class)
from miniai.activations import * # Activation functions
from miniai.init import *        # Weight initialization
from miniai.sgd import *         # Optimizers
from miniai.resnet import *      # ResNet components
from miniai.augment import *     # Data augmentation
from miniai.accel import *       # Mixed precision training

In [ ]:
# Additional imports for progress bars and diffusion schedulers
from fastprogress import progress_bar  # Nice progress bars during training/sampling

# Diffusers library - HuggingFace's diffusion model library
# We import these for reference (we'll build our own versions)
from diffusers import (
    UNet2DModel,     # Pre-built U-Net for 2D images
    DDIMPipeline,    # DDIM sampling pipeline
    DDPMPipeline,    # DDPM sampling pipeline
    DDIMScheduler,   # DDIM noise scheduler
    DDPMScheduler    # DDPM noise scheduler
)

In [ ]:
# Display and reproducibility settings
torch.set_printoptions(
    precision=5,    # Show 5 decimal places
    linewidth=140,  # Wide lines for tensor display
    sci_mode=False  # Don't use scientific notation
)
torch.manual_seed(1)  # Set random seed for reproducibility

# Matplotlib settings
mpl.rcParams['image.cmap'] = 'gray_r'  # Reversed grayscale colormap
mpl.rcParams['figure.dpi'] = 70        # Figure resolution

# Suppress warning messages from libraries
import logging
logging.disable(logging.WARNING)

# Set seed for all random number generators and limit CPU workers
set_seed(42)
if fc.defaults.cpus > 8: 
    fc.defaults.cpus = 8  # Limit CPU workers to prevent memory issues

---
## Section 2: Dataset Loading - Fashion MNIST

We'll use **Fashion MNIST** for training our diffusion model. It contains 70,000 grayscale images of clothing items (28x28 pixels), which we'll pad to 32x32.

### Why Fashion MNIST?
- Small enough for fast experimentation
- More interesting than regular MNIST digits
- 10 classes: T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot

In [ ]:
# Dataset configuration
xl, yl = 'image', 'label'  # Column names in the dataset
name = "fashion_mnist"     # Dataset name on HuggingFace
n_steps = 1000             # Number of diffusion steps (not used directly here)
bs = 512                   # Batch size for training

# Load the Fashion MNIST dataset from HuggingFace
# This downloads the dataset if not already cached
dsd = load_dataset(name)
# dsd contains 'train' (60,000 images) and 'test' (10,000 images) splits

### Signal-to-Noise Ratio Parameter

`sig_data` represents the standard deviation of the training data. This is used in the Karras et al. noise scaling formulation to properly balance the signal and noise during training.

In [ ]:
# Standard deviation of the data distribution
# Fashion MNIST images normalized to [-1, 1] have roughly this std
sig_data = 0.66

---
## Section 3: Data Preprocessing and Noise Functions

For diffusion models, we need to:
1. **Transform images** to tensors and normalize to [-1, 1]
2. **Add noise** at random levels during training
3. **Compute scaling factors** for the network's input/output

### The Karras Scaling Formulation

From the "Elucidating the Design Space of Diffusion-Based Generative Models" paper by Karras et al., we use specific scaling factors:

- **c_skip**: How much of the input to pass through directly
- **c_out**: How to scale the network output
- **c_in**: How to scale the network input

These scalings ensure the network sees normalized inputs and produces outputs in the right range.

In [ ]:
@inplace  # Decorator that modifies the input dictionary in-place
def transformi(b):
    """
    Transform a batch of images for diffusion training.
    
    Args:
        b: Dictionary containing 'image' key with PIL images
    
    Steps:
        1. Convert PIL image to tensor (values in [0, 1])
        2. Pad from 28x28 to 32x32 (add 2 pixels on each side)
        3. Scale from [0, 1] to [-1, 1] using: x*2-1
    """
    # TF.to_tensor: Converts PIL Image to tensor with values [0, 1]
    # F.pad((2,2,2,2)): Adds 2 pixels padding on left, right, top, bottom
    # *2-1: Rescales from [0,1] to [-1,1] (centered at 0)
    b[xl] = [F.pad(TF.to_tensor(o), (2,2,2,2))*2-1 for o in b[xl]]


def scalings(sig):
    """
    Compute the Karras scaling factors for a given noise level.
    
    Args:
        sig: Noise level (sigma) - can be a tensor of values
    
    Returns:
        c_skip: Skip connection weight (how much of input to pass through)
        c_out: Output scaling (how to scale network output)
        c_in: Input scaling (how to scale network input)
    
    The math:
        totvar = sig^2 + sig_data^2  (total variance)
        c_skip = sig_data^2 / totvar  (high when noise is low)
        c_out = sig * sig_data / sqrt(totvar)  (balanced scaling)
        c_in = 1 / sqrt(totvar)  (normalize input variance)
    """
    totvar = sig**2 + sig_data**2  # Total variance = noise variance + data variance
    # Return: c_skip, c_out, c_in
    return sig_data**2/totvar, sig*sig_data/totvar.sqrt(), 1/totvar.sqrt()


def noisify(x0):
    """
    Add noise to clean images for training.
    
    Args:
        x0: Clean images tensor of shape (batch, channels, height, width)
    
    Returns:
        (noised_input, sigma): Tuple of (scaled noisy input, noise levels)
        target: What the network should predict
    
    The noise level sigma is sampled from a log-normal distribution:
        log(sigma) ~ Normal(-1.2, 1.2)
    This gives a good range of noise levels for training.
    """
    device = x0.device
    
    # Sample noise levels from log-normal distribution
    # randn gives N(0,1), we shift to N(-1.2, 1.2) then exponentiate
    sig = (torch.randn([len(x0)])*1.2 - 1.2).exp().to(x0).reshape(-1,1,1,1)
    
    # Generate Gaussian noise with same shape as input
    noise = torch.randn_like(x0, device=device)
    
    # Get scaling factors for this noise level
    c_skip, c_out, c_in = scalings(sig)
    
    # Create noisy image: x_noisy = x_clean + noise * sigma
    noised_input = x0 + noise * sig
    
    # Compute target: what the network should predict
    # This is derived from: output = (x0 - c_skip * noised_input) / c_out
    target = (x0 - c_skip * noised_input) / c_out
    
    # Return scaled input and sigma, plus the target
    return (noised_input * c_in, sig.squeeze()), target


def collate_ddpm(b):
    """
    Custom collate function for the DataLoader.
    Takes a batch of samples, collates them, extracts images, and adds noise.
    
    Args:
        b: List of samples from the dataset
    
    Returns:
        Tuple of ((noisy_images, sigmas), targets) ready for training
    """
    return noisify(default_collate(b)[xl])  # Collate, get images, add noise


def dl_ddpm(ds):
    """
    Create a DataLoader for diffusion training.
    
    Args:
        ds: Dataset to load from
    
    Returns:
        DataLoader with our custom collate function
    """
    return DataLoader(
        ds, 
        batch_size=bs,           # Batch size (512)
        collate_fn=collate_ddpm, # Our custom collate with noise
        num_workers=0            # No multiprocessing (simpler)
    )

In [ ]:
# Apply transforms and create data loaders
tds = dsd.with_transform(transformi)  # Apply our transform to the dataset

# Create DataLoaders for training and testing
# DataLoaders is a convenience class holding train and validation loaders
dls = DataLoaders(
    dl_ddpm(tds['train']),  # Training data loader
    dl_ddpm(tds['test'])    # Validation data loader
)

---
## Section 4: Basic U-Net Architecture (Without Timestep Conditioning)

We'll first build a simple U-Net without timestep information. This helps understand the architecture before adding the complexity of time conditioning.

### U-Net Structure

```
Input (32x32)
    |
    v
[Encoder: Downsample path]
    Down Block 1 (32 channels) -> save features
    Down Block 2 (64 channels) -> save features  
    Down Block 3 (128 channels) -> save features
    Down Block 4 (256 channels) -> save features
    |
    v
[Bottleneck: Middle block]
    |
    v
[Decoder: Upsample path]
    Up Block 4 + saved features from Down 4
    Up Block 3 + saved features from Down 3
    Up Block 2 + saved features from Down 2
    Up Block 1 + saved features from Down 1
    |
    v
Output (32x32)
```

### Pre-activation Convolution Block

We use **pre-activation** order: Norm -> Activation -> Conv (instead of Conv -> Norm -> Activation). This is common in modern architectures and can improve training.

In [ ]:
def unet_conv(ni, nf, ks=3, stride=1, act=nn.SiLU, norm=None, bias=True):
    """
    Create a convolution block with pre-activation order: Norm -> Act -> Conv
    
    Args:
        ni: Number of input channels
        nf: Number of output channels (filters)
        ks: Kernel size (default 3x3)
        stride: Convolution stride (default 1, no downsampling)
        act: Activation function class (default SiLU/Swish)
        norm: Normalization layer class (default None)
        bias: Whether to include bias in conv (default True)
    
    Returns:
        nn.Sequential containing the layers
    
    SiLU (Sigmoid Linear Unit) = x * sigmoid(x)
    Also known as Swish activation - smooth, non-monotonic, works well in practice
    """
    layers = nn.Sequential()
    
    # Pre-activation: add norm first if specified
    if norm: 
        layers.append(norm(ni))  # Normalize input channels
    
    # Pre-activation: add activation before conv
    if act: 
        layers.append(act())  # Instantiate activation function
    
    # The convolution itself
    layers.append(nn.Conv2d(
        ni, nf,                    # Input and output channels
        stride=stride,             # Stride for downsampling
        kernel_size=ks,            # 3x3 kernel
        padding=ks//2,             # Same padding (output size = input size / stride)
        bias=bias                  # Include bias term
    ))
    
    return layers

### Residual Block for U-Net

The **ResBlock** adds skip connections within the block itself. The output is: `output = convs(x) + identity(x)`

This helps with gradient flow and allows the network to learn residual functions.

In [ ]:
class UnetResBlock(nn.Module):
    """
    Residual block for U-Net with two convolutions and a skip connection.
    
    Structure:
        x --> [Conv1] --> [Conv2] --> (+) --> output
        |                              ^
        +-----[Identity/1x1 Conv]------+
    """
    
    def __init__(self, ni, nf=None, ks=3, act=nn.SiLU, norm=nn.BatchNorm2d):
        """
        Args:
            ni: Number of input channels
            nf: Number of output channels (defaults to ni if not specified)
            ks: Kernel size for convolutions
            act: Activation function class
            norm: Normalization layer class
        """
        super().__init__()
        
        # If output channels not specified, keep same as input
        if nf is None: 
            nf = ni
        
        # Main path: two convolutions
        self.convs = nn.Sequential(
            unet_conv(ni, nf, ks, act=act, norm=norm),  # First conv: ni -> nf
            unet_conv(nf, nf, ks, act=act, norm=norm)   # Second conv: nf -> nf
        )
        
        # Skip connection: identity if channels match, 1x1 conv otherwise
        # fc.noop is a function that returns its input unchanged
        self.idconv = fc.noop if ni == nf else nn.Conv2d(ni, nf, 1)

    def forward(self, x):
        # Add main path and skip connection
        return self.convs(x) + self.idconv(x)

### Understanding Python's Method Resolution Order (MRO)

Before building our SaveModule mixin, let's understand how Python handles multiple inheritance. This is important for our save-and-forward pattern.

In [ ]:
# Demonstrating Python's Method Resolution Order (MRO)
# When a class inherits from multiple parents, Python uses MRO to decide
# which method to call

class A:
    def __call__(self):
        # super().__call__() calls the next class in MRO
        super().__call__()
        print('a')

class B:
    def __call__(self): 
        print('b')

class C(A, B): 
    pass  # C inherits from both A and B

# MRO for C is: C -> A -> B -> object
# When C() is called:
# 1. C has no __call__, so A.__call__ is used
# 2. A.__call__ calls super().__call__() which is B.__call__
# 3. B.__call__ prints 'b'
# 4. A.__call__ then prints 'a'

In [ ]:
# Test the MRO
C()()  # Creates C instance and calls it
# Output: 'b' then 'a' (B runs first due to super() call order)

### SaveModule Mixin - Saving Intermediate Features

For U-Net skip connections, we need to **save the output of encoder layers** so the decoder can use them. The `SaveModule` mixin automatically saves the output of any module it's combined with.

In [ ]:
class SaveModule:
    """
    Mixin class that saves the output of forward() to self.saved.
    
    Use this with multiple inheritance to add saving behavior to any module:
        class SavedConv(SaveModule, nn.Conv2d): pass
    
    The MRO ensures SaveModule.forward runs first, calls super().forward()
    to get the actual output, saves it, then returns it.
    """
    
    def forward(self, x, *args, **kwargs):
        # Call the actual forward method (from the other parent class)
        self.saved = super().forward(x, *args, **kwargs)
        return self.saved


# Create saved versions of our blocks
class SavedResBlock(SaveModule, UnetResBlock): 
    """ResBlock that saves its output for skip connections."""
    pass

class SavedConv(SaveModule, nn.Conv2d): 
    """Conv2d that saves its output for skip connections."""
    pass

### Down Block (Encoder)

Each down block:
1. Applies one or more ResBlocks
2. Optionally downsamples with stride-2 convolution
3. Saves outputs for skip connections

In [ ]:
def down_block(ni, nf, add_down=True, num_layers=1):
    """
    Create a downsampling block for the U-Net encoder.
    
    Args:
        ni: Number of input channels
        nf: Number of output channels
        add_down: Whether to add downsampling (stride-2 conv)
        num_layers: Number of ResBlocks in this down block
    
    Returns:
        nn.Sequential containing SavedResBlocks and optional downsampling
    """
    # Create ResBlocks - first one changes channels, rest keep nf
    res = nn.Sequential(*[
        SavedResBlock(
            ni=ni if i==0 else nf,  # First block: ni->nf, others: nf->nf
            nf=nf
        )
        for i in range(num_layers)
    ])
    
    # Add downsampling if requested (halves spatial dimensions)
    if add_down: 
        res.append(SavedConv(
            nf, nf,           # Keep same channels
            3,                # 3x3 kernel
            stride=2,         # Stride 2 halves dimensions
            padding=1         # Padding to maintain proper output size
        ))
    
    return res

### Upsample Function

Upsampling doubles the spatial dimensions. We use nearest-neighbor upsampling followed by a convolution to smooth the result.

In [ ]:
def upsample(nf):
    """
    Create an upsampling block that doubles spatial dimensions.
    
    Args:
        nf: Number of channels (preserved)
    
    Returns:
        nn.Sequential with Upsample (2x) followed by 3x3 conv
    
    Why conv after upsample?
    - Nearest neighbor upsampling creates blocky artifacts
    - The conv smooths these out and learns appropriate blending
    """
    return nn.Sequential(
        nn.Upsample(scale_factor=2.),  # Double height and width (nearest neighbor)
        nn.Conv2d(nf, nf, 3, padding=1)  # 3x3 conv to smooth
    )

### Up Block (Decoder)

Each up block:
1. Concatenates with saved features from encoder (skip connections)
2. Applies ResBlocks
3. Optionally upsamples

In [ ]:
class UpBlock(nn.Module):
    """
    Upsampling block for the U-Net decoder.
    
    Takes features from previous decoder layer AND saved encoder features,
    concatenates them, processes with ResBlocks, then upsamples.
    """
    
    def __init__(self, ni, prev_nf, nf, add_up=True, num_layers=2):
        """
        Args:
            ni: Channels from corresponding encoder level (for skip connection)
            prev_nf: Channels from previous decoder level
            nf: Output channels for this level
            add_up: Whether to add upsampling
            num_layers: Number of ResBlocks
        """
        super().__init__()
        
        # ResBlocks that process concatenated features
        # Channel calculation is tricky due to concatenation:
        # - First resnet: prev_nf (from decoder) + ni (from encoder) -> nf
        # - Other resnets: nf + nf (previous saved) -> nf
        self.resnets = nn.ModuleList([
            UnetResBlock(
                # Input channels: previous decoder output + skip connection
                (prev_nf if i==0 else nf) + (ni if (i==num_layers-1) else nf), 
                nf  # Output channels
            )
            for i in range(num_layers)
        ])
        
        # Upsampling or identity
        self.up = upsample(nf) if add_up else nn.Identity()

    def forward(self, x, ups):
        """
        Args:
            x: Features from previous decoder layer
            ups: List of saved encoder features (we pop from this)
        """
        # Process each resnet, concatenating with saved features
        for resnet in self.resnets: 
            # Concatenate current features with saved encoder features
            # ups.pop() removes and returns the last element
            x = resnet(torch.cat([x, ups.pop()], dim=1))
        
        # Upsample (or identity)
        return self.up(x)

### Complete U-Net Model (No Timesteps)

Now we assemble the complete U-Net. This version doesn't use timestep information - it just predicts the denoised output from the noisy input.

In [ ]:
class UNet2DModel(nn.Module):
    """
    Complete U-Net for image-to-image tasks (e.g., denoising).
    
    Architecture:
        1. Initial conv to expand channels
        2. Encoder (down blocks) - progressively downsample
        3. Middle block - bottleneck processing
        4. Decoder (up blocks) - progressively upsample with skip connections
        5. Final conv to get output channels
    """
    
    def __init__(self, in_channels=3, out_channels=3, nfs=(224,448,672,896), num_layers=1):
        """
        Args:
            in_channels: Input image channels (1 for grayscale, 3 for RGB)
            out_channels: Output image channels
            nfs: Tuple of channel counts for each level
            num_layers: Number of ResBlocks per down block
        """
        super().__init__()
        
        # Initial convolution to expand to first feature size
        self.conv_in = nn.Conv2d(in_channels, nfs[0], kernel_size=3, padding=1)
        
        # Build encoder (downsampling path)
        nf = nfs[0]  # Current number of features
        self.downs = nn.Sequential()
        
        for i in range(len(nfs)):
            ni = nf          # Input channels from previous level
            nf = nfs[i]      # Output channels for this level
            # Don't downsample at the last level
            self.downs.append(down_block(ni, nf, add_down=i!=len(nfs)-1, num_layers=num_layers))
        
        # Middle block (bottleneck)
        self.mid_block = UnetResBlock(nfs[-1])

        # Build decoder (upsampling path)
        rev_nfs = list(reversed(nfs))  # Reverse channel order for decoder
        nf = rev_nfs[0]
        self.ups = nn.ModuleList()
        
        for i in range(len(nfs)):
            prev_nf = nf                              # Previous decoder level channels
            nf = rev_nfs[i]                           # Current level channels
            ni = rev_nfs[min(i+1, len(nfs)-1)]        # Encoder skip connection channels
            # Don't upsample at the last level
            self.ups.append(UpBlock(ni, prev_nf, nf, add_up=i!=len(nfs)-1, num_layers=num_layers+1))
        
        # Final convolution to get output channels
        self.conv_out = unet_conv(nfs[0], out_channels, act=nn.SiLU, norm=nn.BatchNorm2d)

    def forward(self, inp):
        """
        Args:
            inp: Tuple of (noisy_image, sigma) - we only use the image here
        """
        # Initial convolution
        x = self.conv_in(inp[0])  # inp[0] is the image, inp[1] is sigma (unused here)
        
        # Save initial features for skip connections
        saved = [x]
        
        # Encoder path - each block saves its outputs in saved list
        x = self.downs(x)
        
        # Collect all saved features from down blocks
        saved += [p.saved for o in self.downs for p in o]
        
        # Middle block
        x = self.mid_block(x)
        
        # Decoder path - uses saved features via skip connections
        for block in self.ups: 
            x = block(x, saved)
        
        # Final convolution
        return self.conv_out(x)

In [ ]:
# Create the model
# For Fashion MNIST (grayscale), we use 1 input/output channel
# nfs=(32,64,128,256) gives us 4 levels with increasing channels
model = UNet2DModel(
    in_channels=1,              # Grayscale input
    out_channels=1,             # Grayscale output
    nfs=(32, 64, 128, 256),     # Channel progression
    num_layers=2                # 2 ResBlocks per down block
)

### Training the Basic U-Net

In [ ]:
# Training hyperparameters
lr = 3e-3       # Learning rate
epochs = 25     # Number of training epochs

# Optimizer: Adam with small epsilon for numerical stability
opt_func = partial(optim.Adam, eps=1e-5)

# Learning rate scheduler: OneCycleLR
# Total steps = epochs * batches_per_epoch
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)

# Callbacks for training
cbs = [
    DeviceCB(),           # Move data/model to GPU
    MixedPrecision(),     # FP16 for faster training
    ProgressCB(plot=True), # Progress bar with loss plot
    MetricsCB(),          # Track metrics
    BatchSchedCB(sched)   # Update LR each batch
]

# Create the Learner
learn = Learner(
    model, 
    dls, 
    nn.MSELoss(),  # Mean Squared Error loss for regression
    lr=lr, 
    cbs=cbs, 
    opt_func=opt_func
)

In [ ]:
# Train the model (this will take some time)
learn.fit(epochs)

---
## Section 5: Timestep Embeddings

The basic U-Net doesn't know what noise level it's working with. For better results, we need to **condition the network on the timestep/noise level**.

### Why Timestep Conditioning?

- At **high noise levels**, the network should focus on global structure
- At **low noise levels**, the network should focus on fine details
- Without knowing the noise level, the network can't adjust its behavior

### Sinusoidal Positional Embeddings

We use **sinusoidal embeddings** (same as in Transformers) to encode the timestep:

$$\text{embed}(t, 2i) = \sin(t \cdot \omega_i)$$
$$\text{embed}(t, 2i+1) = \cos(t \cdot \omega_i)$$

Where $\omega_i = \exp\left(-\frac{\log(\text{max_period}) \cdot i}{d/2}\right)$

This gives each timestep a unique embedding that varies smoothly.

In [ ]:
# Parameters for timestep embedding
emb_dim = 16       # Embedding dimension
tsteps = torch.linspace(-10, 10, 100)  # Test timesteps from -10 to 10
max_period = 10000  # Controls the frequency range

In [ ]:
# log(10000) ≈ 9.21
# This sets the range of frequencies in our embedding
math.log(10000)

In [ ]:
# Compute the frequency exponents
# This creates a range from 1 to 1/max_period for different embedding dimensions
# linspace(0, 1, emb_dim//2) gives [0, ..., 1] with emb_dim/2 values
# After exp(), we get frequencies from 1 to 1/10000
exponent = -math.log(max_period) * torch.linspace(0, 1, emb_dim//2, device=tsteps.device)

In [ ]:
# Visualize the exponents (log scale frequencies)
plt.plot(exponent)
plt.title('Frequency Exponents')
plt.xlabel('Dimension')
plt.ylabel('Exponent');

In [ ]:
# Compute the raw embeddings (before sin/cos)
# tsteps[:,None] shape: (100, 1)
# exponent.exp()[None,:] shape: (1, 8)
# Result shape: (100, 8) - each timestep gets 8 values
emb = tsteps[:,None].float() * exponent.exp()[None,:]
emb.shape  # (num_timesteps, emb_dim/2)

In [ ]:
# Visualize embeddings for different timesteps
# Each line shows the 8 raw values for one timestep
plt.plot(emb[0], label='t=-10')
plt.plot(emb[10], label='t=-8')
plt.plot(emb[20], label='t=-6')
plt.plot(emb[50], label='t=0')
plt.plot(emb[-1], label='t=10')
plt.legend()
plt.title('Raw Embeddings at Different Timesteps')
plt.xlabel('Dimension')
plt.ylabel('Value');

In [ ]:
# Apply sin and cos to get final embeddings
# Concatenate sin and cos along last dimension to double the embedding size
emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
emb.shape  # (100, 16) - full embedding dimension

In [ ]:
# Visualize how individual embedding dimensions vary across timesteps
# Each line is one dimension of the embedding
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(emb[:,0], label='dim 0 (sin, high freq)')
plt.plot(emb[:,1], label='dim 1')
plt.plot(emb[:,2], label='dim 2')
plt.plot(emb[:,3], label='dim 3')
plt.plot(emb[:,4], label='dim 4')
plt.legend()
plt.title('First 5 Dimensions')
plt.xlabel('Timestep Index');

plt.subplot(1, 2, 2)
plt.plot(emb[:,8], label='dim 8 (cos, high freq)')
plt.plot(emb[:,9], label='dim 9')
plt.plot(emb[:,10], label='dim 10')
plt.legend()
plt.title('Cosine Dimensions')
plt.xlabel('Timestep Index');

In [ ]:
# Visualize the entire embedding matrix
# Rows are timesteps, columns are embedding dimensions
# Pattern shows how each timestep has a unique fingerprint
show_image(emb.T, figsize=(7,7));

### Timestep Embedding Function

Now let's package this into a reusable function.

In [ ]:
def timestep_embedding(tsteps, emb_dim, max_period=10000):
    """
    Create sinusoidal timestep embeddings.
    
    Args:
        tsteps: Tensor of timestep values, shape (batch_size,)
        emb_dim: Dimension of the output embedding
        max_period: Controls the frequency range (default 10000)
    
    Returns:
        Embeddings tensor, shape (batch_size, emb_dim)
    
    Each timestep gets a unique embedding that varies smoothly.
    Lower dimensions have high frequencies, higher dimensions have low frequencies.
    """
    # Compute frequency exponents: from 1 to 1/max_period
    exponent = -math.log(max_period) * torch.linspace(0, 1, emb_dim//2, device=tsteps.device)
    
    # Multiply timesteps by frequencies
    # tsteps[:,None]: (batch, 1)
    # exponent.exp()[None,:]: (1, emb_dim/2)
    # Result: (batch, emb_dim/2)
    emb = tsteps[:,None].float() * exponent.exp()[None,:]
    
    # Apply sin and cos, concatenate
    emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
    
    # Handle odd embedding dimensions by padding with zero
    return F.pad(emb, (0,1,0,0)) if emb_dim%2==1 else emb

In [ ]:
# Visualize with different max_period values
# Smaller max_period = higher frequencies = faster oscillation
show_image(timestep_embedding(tsteps, 32, max_period=1000).T, figsize=(7,7));
plt.title('max_period=1000 (faster oscillation)');

In [ ]:
# Even smaller max_period
show_image(timestep_embedding(tsteps, 32, max_period=10).T, figsize=(7,7));
plt.title('max_period=10 (very fast oscillation)');

---
## Section 6: U-Net with Timestep Conditioning

Now we'll build a U-Net that uses timestep information. The key changes are:

1. **Timestep MLP**: Convert timestep embedding to a useful representation
2. **Modified ResBlocks**: Inject timestep information via scale and shift (FiLM conditioning)

### FiLM Conditioning (Feature-wise Linear Modulation)

We inject the timestep by computing scale ($\gamma$) and shift ($\beta$) from the timestep embedding:

$$\text{output} = \gamma \cdot \text{features} + \beta$$

This lets the network adjust its activations based on the noise level.

In [ ]:
from functools import wraps  # For preserving function metadata in decorators

In [ ]:
def lin(ni, nf, act=nn.SiLU, norm=None, bias=True):
    """
    Create a linear layer with pre-activation order: Norm -> Act -> Linear
    
    Args:
        ni: Number of input features
        nf: Number of output features
        act: Activation function class
        norm: Normalization layer class (e.g., BatchNorm1d)
        bias: Whether to include bias
    
    Returns:
        nn.Sequential containing the layers
    """
    layers = nn.Sequential()
    if norm: layers.append(norm(ni))  # Normalize input
    if act:  layers.append(act())      # Activation
    layers.append(nn.Linear(ni, nf, bias=bias))  # Linear transformation
    return layers

### Embedding-Conditioned ResBlock

This ResBlock takes an additional timestep embedding input and uses it to modulate the features.

In [ ]:
class EmbResBlock(nn.Module):
    """
    ResBlock conditioned on timestep embedding using FiLM.
    
    Structure:
        x -----> [Conv1] ----> [Scale/Shift by t] ----> [Conv2] ----> (+) --> out
        |                                                              ^
        +---------------------[Identity/1x1 Conv]-----------------------+
        
        t -----> [MLP] ----> scale, shift
    """
    
    def __init__(self, n_emb, ni, nf=None, ks=3, act=nn.SiLU, norm=nn.BatchNorm2d):
        """
        Args:
            n_emb: Dimension of the timestep embedding
            ni: Number of input channels
            nf: Number of output channels (defaults to ni)
            ks: Kernel size
            act: Activation function
            norm: Normalization layer
        """
        super().__init__()
        if nf is None: 
            nf = ni
        
        # Project timestep embedding to scale and shift parameters
        # Output is 2*nf: first half is scale, second half is shift
        self.emb_proj = nn.Linear(n_emb, nf * 2)
        
        # First conv: ni -> nf
        self.conv1 = unet_conv(ni, nf, ks, act=act, norm=norm)
        
        # Second conv: nf -> nf (after scale/shift)
        self.conv2 = unet_conv(nf, nf, ks, act=act, norm=norm)
        
        # Skip connection
        self.idconv = fc.noop if ni == nf else nn.Conv2d(ni, nf, 1)

    def forward(self, x, t):
        """
        Args:
            x: Input features, shape (batch, channels, height, width)
            t: Timestep embedding, shape (batch, n_emb)
        """
        inp = x  # Save for skip connection
        
        # First convolution
        x = self.conv1(x)
        
        # Get scale and shift from timestep embedding
        # F.silu applies SiLU activation to embedding before projection
        emb = self.emb_proj(F.silu(t))[:, :, None, None]  # (batch, 2*nf, 1, 1)
        
        # Split into scale and shift
        scale, shift = torch.chunk(emb, 2, dim=1)  # Each is (batch, nf, 1, 1)
        
        # Apply FiLM: x = x * (1 + scale) + shift
        # This modulates features based on timestep
        x = x * (1 + scale) + shift
        
        # Second convolution
        x = self.conv2(x)
        
        # Add skip connection
        return x + self.idconv(inp)

### Saved Function Decorator

For the timestep-conditioned model, we need a different approach to saving features. Instead of a mixin class, we use a decorator that wraps the forward method.

In [ ]:
def saved(m, blk):
    """
    Wrap a module's forward method to save its output.
    
    Args:
        m: Module to wrap
        blk: Block that will store the saved outputs (must have .saved attribute)
    
    Returns:
        The modified module (same object, modified in place)
    
    This uses a closure to capture the original forward method and the block.
    """
    m_ = m.forward  # Store original forward method

    @wraps(m.forward)  # Preserve function metadata
    def _f(*args, **kwargs):
        res = m_(*args, **kwargs)  # Call original forward
        blk.saved.append(res)       # Save output to block's list
        return res

    m.forward = _f  # Replace forward with our wrapper
    return m

### Down Block with Timestep Conditioning

In [ ]:
class DownBlock(nn.Module):
    """
    Downsampling block with timestep conditioning.
    
    Contains EmbResBlocks that process features conditioned on timestep,
    plus optional downsampling convolution.
    """
    
    def __init__(self, n_emb, ni, nf, add_down=True, num_layers=1):
        """
        Args:
            n_emb: Timestep embedding dimension
            ni: Input channels
            nf: Output channels
            add_down: Whether to downsample
            num_layers: Number of ResBlocks
        """
        super().__init__()
        
        # ResBlocks with embedding conditioning
        # Each is wrapped with 'saved' to store outputs for skip connections
        self.resnets = nn.ModuleList([
            saved(EmbResBlock(n_emb, ni if i==0 else nf, nf), self)
            for i in range(num_layers)
        ])
        
        # Downsampling conv (also saved) or identity
        self.down = saved(nn.Conv2d(nf, nf, 3, stride=2, padding=1), self) if add_down else nn.Identity()

    def forward(self, x, t):
        """
        Args:
            x: Input features
            t: Timestep embedding
        """
        self.saved = []  # Reset saved list for this forward pass
        
        # Apply each ResBlock with timestep conditioning
        for resnet in self.resnets: 
            x = resnet(x, t)
        
        # Downsample (or identity)
        x = self.down(x)
        return x

### Up Block with Timestep Conditioning

In [ ]:
class UpBlock(nn.Module):
    """
    Upsampling block with timestep conditioning and skip connections.
    """
    
    def __init__(self, n_emb, ni, prev_nf, nf, add_up=True, num_layers=2):
        """
        Args:
            n_emb: Timestep embedding dimension
            ni: Channels from encoder (for skip connections)
            prev_nf: Channels from previous decoder level
            nf: Output channels
            add_up: Whether to upsample
            num_layers: Number of ResBlocks
        """
        super().__init__()
        
        # ResBlocks processing concatenated features
        self.resnets = nn.ModuleList([
            EmbResBlock(
                n_emb,
                (prev_nf if i==0 else nf) + (ni if (i==num_layers-1) else nf),
                nf
            )
            for i in range(num_layers)
        ])
        
        # Upsampling or identity
        self.up = upsample(nf) if add_up else nn.Identity()

    def forward(self, x, t, ups):
        """
        Args:
            x: Features from previous decoder level
            t: Timestep embedding
            ups: List of saved encoder features
        """
        # Process each ResBlock with skip connections and timestep
        for resnet in self.resnets: 
            x = resnet(torch.cat([x, ups.pop()], dim=1), t)
        
        return self.up(x)

### Complete U-Net with Timestep Embedding

In [ ]:
class EmbUNetModel(nn.Module):
    """
    U-Net conditioned on timestep embeddings.
    
    This is the main architecture for diffusion models:
    1. Encode timestep as sinusoidal embedding
    2. Process embedding through MLP
    3. Inject embedding into all ResBlocks via FiLM conditioning
    """
    
    def __init__(self, in_channels=3, out_channels=3, nfs=(224,448,672,896), num_layers=1):
        """
        Args:
            in_channels: Input image channels
            out_channels: Output image channels
            nfs: Channel counts for each level
            num_layers: ResBlocks per down block
        """
        super().__init__()
        
        # Initial convolution
        self.conv_in = nn.Conv2d(in_channels, nfs[0], kernel_size=3, padding=1)
        
        # Timestep embedding dimensions
        self.n_temb = nf = nfs[0]  # Base timestep embedding dimension
        n_emb = nf * 4              # Expanded embedding dimension (4x)
        
        # MLP to process timestep embedding
        # Takes sinusoidal embedding, outputs rich feature representation
        # TODO comment in original: remove act func from 1st MLP layer
        self.emb_mlp = nn.Sequential(
            lin(self.n_temb, n_emb, norm=nn.BatchNorm1d),  # n_temb -> n_emb
            lin(n_emb, n_emb)                               # n_emb -> n_emb
        )
        
        # Encoder (downsampling path)
        self.downs = nn.ModuleList()
        for i in range(len(nfs)):
            ni = nf
            nf = nfs[i]
            self.downs.append(DownBlock(
                n_emb, ni, nf, 
                add_down=i!=len(nfs)-1, 
                num_layers=num_layers
            ))
        
        # Middle block
        self.mid_block = EmbResBlock(n_emb, nfs[-1])

        # Decoder (upsampling path)
        rev_nfs = list(reversed(nfs))
        nf = rev_nfs[0]
        self.ups = nn.ModuleList()
        for i in range(len(nfs)):
            prev_nf = nf
            nf = rev_nfs[i]
            ni = rev_nfs[min(i+1, len(nfs)-1)]
            self.ups.append(UpBlock(
                n_emb, ni, prev_nf, nf, 
                add_up=i!=len(nfs)-1, 
                num_layers=num_layers+1
            ))
        
        # Final convolution (no bias to let BatchNorm handle mean)
        self.conv_out = unet_conv(nfs[0], out_channels, act=nn.SiLU, norm=nn.BatchNorm2d, bias=False)

    def forward(self, inp):
        """
        Args:
            inp: Tuple of (noisy_image, sigma)
                noisy_image: (batch, channels, height, width)
                sigma: (batch,) noise levels
        """
        x, t = inp  # Unpack input
        
        # Create timestep embedding from sigma values
        temb = timestep_embedding(t, self.n_temb)
        
        # Process through MLP to get rich embedding
        emb = self.emb_mlp(temb)
        
        # Initial convolution
        x = self.conv_in(x)
        saved = [x]  # Start collecting features for skip connections
        
        # Encoder path
        for block in self.downs: 
            x = block(x, emb)
        
        # Collect all saved features
        saved += [p for o in self.downs for p in o.saved]
        
        # Middle block
        x = self.mid_block(x, emb)
        
        # Decoder path with skip connections
        for block in self.ups: 
            x = block(x, emb, saved)
        
        # Final convolution
        return self.conv_out(x)

In [ ]:
# Create the timestep-conditioned model
model = EmbUNetModel(
    in_channels=1,           # Grayscale
    out_channels=1,          # Grayscale output
    nfs=(32, 64, 128, 256),  # Channel progression
    num_layers=2             # ResBlocks per down block
)

### Training the Timestep-Conditioned U-Net

In [ ]:
# Training hyperparameters (higher LR since we have timestep info)
lr = 1e-2       # Higher learning rate than before
epochs = 25

opt_func = partial(optim.Adam, eps=1e-5)
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)

# Callbacks (note: MixedPrecision moved to end)
cbs = [
    DeviceCB(), 
    ProgressCB(plot=True), 
    MetricsCB(), 
    BatchSchedCB(sched), 
    MixedPrecision()
]

# Create fresh model and learner
model = EmbUNetModel(in_channels=1, out_channels=1, nfs=(32,64,128,256), num_layers=2)
learn = Learner(model, dls, nn.MSELoss(), lr=lr, cbs=cbs, opt_func=opt_func)

In [ ]:
# Train the timestep-conditioned model
learn.fit(epochs)

---
## Section 7: Sampling (Generating Images)

After training, we generate images by starting from pure noise and gradually denoising. Different sampling algorithms give different quality/speed tradeoffs.

### The Sampling Process

1. Start with random Gaussian noise at high sigma (e.g., 80)
2. Use the model to predict the denoised image
3. Take a step toward the prediction
4. Repeat until sigma reaches 0

In [ ]:
# Import for image quality evaluation
from miniai.fid import ImageEval

In [ ]:
# Load pre-trained classifier for FID/KID evaluation
# This classifier was trained on Fashion MNIST
cmodel = torch.load('models/data_aug2.pkl')

# Remove the final classification layers (we want features)
del(cmodel[8])  # Remove classifier head
del(cmodel[7])  # Remove adaptive pooling

# Create data loader for real images (for comparison)
bs = 2048
tds2 = dsd.with_transform(transformi)
dls2 = DataLoaders.from_dd(tds, bs, num_workers=fc.defaults.cpus)

dt = dls2.train
xb, yb = next(iter(dt))

# Create image evaluator (computes FID and KID)
ie = ImageEval(cmodel, dls2, cbs=[DeviceCB()])

In [ ]:
# Sample size: generate 2048 images of shape 1x32x32
sz = (2048, 1, 32, 32)

### Karras Sigma Schedule

The Karras et al. paper recommends a specific schedule for sigma values during sampling. The schedule goes from high noise (sigma_max) to zero, with most steps concentrated in the middle range.

In [ ]:
def sigmas_karras(n, sigma_min=0.01, sigma_max=80., rho=7.):
    """
    Generate the Karras sigma schedule.
    
    Args:
        n: Number of sampling steps
        sigma_min: Minimum sigma (almost zero noise)
        sigma_max: Maximum sigma (pure noise)
        rho: Controls the shape of the schedule (7 is recommended)
    
    Returns:
        Tensor of n+1 sigma values from sigma_max to 0
    
    The schedule is designed so that perceptually equal changes
    in image quality correspond to equal steps in the schedule.
    """
    ramp = torch.linspace(0, 1, n)  # Linear ramp from 0 to 1
    
    # Transform using rho exponent
    min_inv_rho = sigma_min ** (1/rho)
    max_inv_rho = sigma_max ** (1/rho)
    
    # Interpolate in transformed space, then transform back
    sigmas = (max_inv_rho + ramp * (min_inv_rho - max_inv_rho)) ** rho
    
    # Append 0 at the end (final step has no noise)
    return torch.cat([sigmas, tensor([0.])]).cuda()


def denoise(model, x, sig):
    """
    Use the model to denoise an image at a given sigma level.
    
    Args:
        model: The trained diffusion U-Net
        x: Noisy image
        sig: Current noise level (scalar)
    
    Returns:
        Denoised image prediction
    
    This applies the Karras scaling factors to get the final prediction.
    """
    sig = sig[None]  # Add batch dimension
    
    # Get scaling factors
    c_skip, c_out, c_in = scalings(sig)
    
    # Model predicts scaled output, we combine with skip connection
    # Final prediction = model_output * c_out + input * c_skip
    return model((x * c_in, sig)) * c_out + x * c_skip

### Euler Ancestral Sampler

The Euler Ancestral sampler adds noise at each step, which can improve diversity but may reduce quality. The `eta` parameter controls the amount of added noise.

In [ ]:
def get_ancestral_step(sigma_from, sigma_to, eta=1.):
    """
    Compute parameters for ancestral sampling step.
    
    Args:
        sigma_from: Current sigma
        sigma_to: Target sigma
        eta: Noise injection parameter (0=deterministic, 1=full stochastic)
    
    Returns:
        sigma_down: Sigma to use for the deterministic step
        sigma_up: Amount of noise to add after the step
    """
    if not eta: 
        return sigma_to, 0.  # No noise injection
    
    var_to, var_from = sigma_to**2, sigma_from**2
    
    # Compute how much noise to add
    sigma_up = min(sigma_to, eta * (var_to * (var_from - var_to) / var_from)**0.5)
    
    # Sigma for deterministic step
    return (var_to - sigma_up**2)**0.5, sigma_up


@torch.no_grad()  # Disable gradients for sampling (inference only)
def sample_euler_ancestral(x, sigs, i, model, eta=1.):
    """
    One step of Euler ancestral sampling.
    
    Args:
        x: Current noisy image
        sigs: Full sigma schedule
        i: Current step index
        model: Diffusion model
        eta: Noise injection parameter
    
    Returns:
        Updated image at next sigma level
    """
    sig, sig2 = sigs[i], sigs[i+1]  # Current and next sigma
    
    # Get denoised prediction
    denoised = denoise(model, x, sig)
    
    # Get step parameters
    sigma_down, sigma_up = get_ancestral_step(sig, sig2, eta=eta)
    
    # Euler step toward denoised image
    x = x + (x - denoised) / sig * (sigma_down - sig)
    
    # Add noise (ancestral part)
    return x + torch.randn_like(x) * sigma_up

### Euler and Heun Samplers

- **Euler**: Simple first-order method (fast but less accurate)
- **Heun**: Second-order method (slower but more accurate)

In [ ]:
@torch.no_grad()
def sample_euler(x, sigs, i, model):
    """
    One step of basic Euler sampling (deterministic).
    
    This is the simplest ODE solver: take a step in the direction
    of the derivative.
    """
    sig, sig2 = sigs[i], sigs[i+1]
    denoised = denoise(model, x, sig)
    
    # Euler step: x_new = x + derivative * step_size
    # derivative = (x - denoised) / sig
    # step_size = sig2 - sig
    return x + (x - denoised) / sig * (sig2 - sig)


@torch.no_grad()
def sample_heun(x, sigs, i, model, s_churn=0., s_tmin=0., s_tmax=float('inf'), s_noise=1.):
    """
    One step of Heun's method (2nd order, more accurate).
    
    Heun's method:
    1. Predict where we'll be after Euler step
    2. Evaluate derivative at that predicted point
    3. Average the two derivatives
    4. Take step using the averaged derivative
    
    Args:
        s_churn: Amount of noise to add for stochasticity
        s_tmin, s_tmax: Sigma range where churn is applied
        s_noise: Scale for churn noise
    """
    sig, sig2 = sigs[i], sigs[i+1]
    n = len(sigs)
    
    # Optionally add "churn" (noise) for diversity
    gamma = min(s_churn/(n-1), 2**0.5 - 1) if s_tmin <= sig <= s_tmax else 0.
    eps = torch.randn_like(x) * s_noise
    sigma_hat = sig * (gamma + 1)
    
    if gamma > 0: 
        x = x + eps * (sigma_hat**2 - sig**2)**0.5
    
    # First derivative
    denoised = denoise(model, x, sig)
    d = (x - denoised) / sig
    
    # Step size
    dt = sig2 - sigma_hat
    
    # Euler step to get x_2
    x_2 = x + d * dt
    
    # If we're at the end, just return Euler result
    if sig2 == 0: 
        return x_2
    
    # Second derivative at predicted point
    denoised_2 = denoise(model, x_2, sig2)
    d_2 = (x_2 - denoised_2) / sig2
    
    # Average derivatives
    d_prime = (d + d_2) / 2
    
    # Take step with averaged derivative
    return x + d_prime * dt

### General Sampling Function

In [ ]:
def sample(sampler, model, steps=100, sigma_max=80., **kwargs):
    """
    Generate images using a given sampler.
    
    Args:
        sampler: Sampling function (sample_euler, sample_heun, etc.)
        model: Trained diffusion model
        steps: Number of sampling steps
        sigma_max: Starting noise level
        **kwargs: Additional arguments for the sampler
    
    Returns:
        List of predictions at each step
    """
    preds = []
    
    # Start with pure Gaussian noise
    x = torch.randn(sz).cuda() * sigma_max
    
    # Get sigma schedule
    sigs = sigmas_karras(steps, sigma_max=sigma_max)
    
    # Sample iteratively
    for i in progress_bar(range(len(sigs)-1)):
        x = sampler(x, sigs, i, model, **kwargs)
        preds.append(x)
    
    return preds

### Linear Multi-Step (LMS) Sampler

LMS uses previous derivative estimates to make better predictions. Higher orders use more history for potentially better accuracy.

In [ ]:
from scipy import integrate  # For numerical integration

In [ ]:
def linear_multistep_coeff(order, t, i, j):
    """
    Compute coefficients for linear multi-step method.
    
    These coefficients determine how to weight previous derivatives
    to estimate the integral from t[i] to t[i+1].
    
    Args:
        order: Order of the method (how many previous steps to use)
        t: Time/sigma schedule
        i: Current step index
        j: Which previous derivative (0=most recent)
    
    Returns:
        Coefficient for the j-th previous derivative
    """
    if order - 1 > i: 
        raise ValueError(f'Order {order} too high for step {i}')
    
    def fn(tau):
        # Lagrange interpolation basis polynomial
        prod = 1.
        for k in range(order):
            if j == k: 
                continue
            prod *= (tau - t[i-k]) / (t[i-j] - t[i-k])
        return prod
    
    # Integrate to get coefficient
    return integrate.quad(fn, t[i], t[i+1], epsrel=1e-4)[0]


@torch.no_grad()
def sample_lms(model, steps=100, order=4, sigma_max=80.):
    """
    Generate images using Linear Multi-Step method.
    
    LMS uses previous derivative estimates to predict the next step
    more accurately than simple Euler.
    
    Args:
        model: Trained diffusion model
        steps: Number of sampling steps
        order: Order of the method (higher = uses more history)
        sigma_max: Starting noise level
    
    Returns:
        List of predictions at each step
    """
    preds = []
    
    # Start with pure noise
    x = torch.randn(sz).cuda() * sigma_max
    
    # Get sigma schedule
    sigs = sigmas_karras(steps, sigma_max=sigma_max)
    
    # Store previous derivatives
    ds = []
    
    for i in progress_bar(range(len(sigs)-1)):
        sig = sigs[i]
        
        # Get current derivative
        denoised = denoise(model, x, sig)
        d = (x - denoised) / sig
        
        # Add to history
        ds.append(d)
        
        # Keep only 'order' previous derivatives
        if len(ds) > order: 
            ds.pop(0)
        
        # Current order (limited by available history)
        cur_order = min(i+1, order)
        
        # Compute coefficients
        coeffs = [linear_multistep_coeff(cur_order, sigs, i, j) for j in range(cur_order)]
        
        # Take weighted step using all available derivatives
        x = x + sum(coeff * d for coeff, d in zip(coeffs, reversed(ds)))
        
        preds.append(x)
    
    return preds

### Generate and Evaluate Samples

In [ ]:
# Generate samples using LMS sampler
# order=3 uses 3 previous derivatives for better accuracy
preds = sample_lms(model, steps=20, order=3)

# Alternative samplers (commented out):
# preds = sample(sample_euler_ancestral, model, steps=100, eta=1.)
# preds = sample(sample_euler, model, steps=100)
# preds = sample(sample_heun, model, steps=20, s_churn=0.5)

In [ ]:
# Check the value range of generated samples
s = preds[-1]  # Final samples
s.min(), s.max()  # Should be roughly in [-1, 1]

In [ ]:
# Visualize some generated samples
# clamp to [-1, 1] for proper display
show_images(s[:25].clamp(-1, 1), imsize=1.5)

In [ ]:
# Evaluate image quality using FID and KID
# Lower is better for both metrics
# FID (Frechet Inception Distance): measures similarity of distributions
# KID (Kernel Inception Distance): similar to FID but unbiased
ie.fid(s), ie.kid(s), s.shape

In [ ]:
# Generate more samples to check consistency
preds = sample_lms(model, steps=20, order=3)
s = preds[-1]
ie.fid(s), ie.kid(s), s.shape

In [ ]:
# Another batch for variance
preds = sample_lms(model, steps=20, order=3)
s = preds[-1]
ie.fid(s), ie.kid(s), s.shape

---
## Summary

In this notebook, we built a complete diffusion U-Net from scratch:

### Architecture Components

1. **Pre-activation Conv Blocks**: Norm -> Activation -> Conv order
2. **ResBlocks with Skip Connections**: `output = convs(x) + identity(x)`
3. **SaveModule Mixin**: Saves outputs for U-Net skip connections
4. **Down/Up Blocks**: Handle channel changes and spatial resampling

### Timestep Conditioning

1. **Sinusoidal Embeddings**: Encode timestep as smooth, unique vectors
2. **FiLM Conditioning**: `x = x * (1 + scale) + shift` based on timestep
3. **Embedding MLP**: Process raw embeddings into rich features

### Sampling Methods

| Method | Order | Stochastic | Speed | Quality |
|--------|-------|------------|-------|--------|
| Euler | 1st | No | Fast | Lower |
| Euler Ancestral | 1st | Yes | Fast | Medium |
| Heun | 2nd | Optional | Medium | Higher |
| LMS | Higher | No | Medium | Highest |

### Key Formulas

**Karras Scaling:**
$$c_{skip} = \frac{\sigma_{data}^2}{\sigma^2 + \sigma_{data}^2}$$
$$c_{out} = \frac{\sigma \cdot \sigma_{data}}{\sqrt{\sigma^2 + \sigma_{data}^2}}$$
$$c_{in} = \frac{1}{\sqrt{\sigma^2 + \sigma_{data}^2}}$$

**Sinusoidal Embedding:**
$$\text{embed}(t, 2i) = \sin(t \cdot \omega_i), \quad \text{embed}(t, 2i+1) = \cos(t \cdot \omega_i)$$

### Quality Metrics

- **FID (Frechet Inception Distance)**: Measures how similar generated distribution is to real distribution
- **KID (Kernel Inception Distance)**: Similar to FID but with unbiased estimation

Lower values = better quality for both metrics.